# Module 1 Fire History Explorer
**Demo region:** Pine Ridge Reservation, South Dakota (Oglala Sioux Tribe)

> **Governance status of this notebook**
>
> Runs on **public federal data only**, at **PUBLIC tier**, under no data-use
> agreement. That is a deliberate default, see Module 4.
>
> Federal data being public makes it legally available. It does not make an
> *analysis about a Nation* publishable by default: the aggregate product is a
> new artifact, and releasing it is a governance decision that source licensing
> does not settle. `ctx.check_publication()` blocks until sign-off is recorded.
>
> Set `NATION` and `REGION_NAME` below to retarget.

## What this module argues

Most fire-history analyses open the satellite record and treat it as the
baseline. This one does not, because for Tribal lands that framing produces a
specific and consequential error.

MTBS begins in 1984, with Landsat 5. That record covers the era of maximum fire
suppression, and it follows roughly a century in which Indigenous cultural
burning was actively suppressed and in places criminalized. So the "historical"
fire frequency it shows is not a natural baseline, it is the signature of a
policy. An analysis that measures departure from it has built the policy into
the reference condition and is then surprised by contemporary fire.

Three steps:

1. Fire perimeters and rotation, 1984–present
2. Ignition records and cause profile, 1992–2020 (Short's FOD)
3. The satellite-era rotation set against a pre-suppression reference with the
   limits of that comparison stated

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import geopandas as gpd

from daear_toolkit import sovereignty as sv
from daear_toolkit import tribal_access as ta
from daear_toolkit import tribal_indicators as ti

NATION = "Oglala Sioux Tribe"
REGION_NAME = "Pine Ridge"

# The governance context every data call is gated through. No agreement is
# recorded, so this context permits PUBLIC tier only attempting anything
# finer raises rather than quietly returning data.
ctx = sv.GovernanceContext(nation=NATION)
print(f"Nation:        {ctx.nation}")
print(f"Agreement:     {ctx.agreement or 'none recorded'}")
print(f"Maximum tier:  {ctx.max_tier().name}")
print(f"\nTier rules at {ctx.max_tier().name}:")
for k, v in sv.TIER_RULES[ctx.max_tier()].items():
    print(f"  {k:<10} {v}")

In [ ]:
import json
geojson = ta._arcgis_query(ta._AIANNH, ta._AIANNH_RESERVATION_LAYER,
                            bbox=None, where=f"UPPER(NAME) LIKE '%{REGION_NAME.upper()}%'")
print(f"Features returned: {len(geojson.get('features', []))}")
if geojson.get('features'):
    print("First feature keys:", geojson['features'][0].keys())
    print("Geometry:", geojson['features'][0].get('geometry'))
    print("Properties:", geojson['features'][0].get('properties'))
else:
    print("Full response:", json.dumps(geojson, indent=2)[:500])

In [ ]:
# What is REGION_NAME set to?
print("REGION_NAME:", REGION_NAME)

# List current layers in the AIANNH service
import requests
r = requests.get(
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/AIANNHA/MapServer",
    params={"f": "json"}, timeout=30
)
for layer in r.json().get("layers", []):
    print(layer["id"], layer["name"])

In [ ]:
boundary = ta.get_tribal_boundary(ctx, name=REGION_NAME)
BBOX = tuple(boundary.total_bounds)
area_ha = float(boundary.to_crs(epsg=5070).area.sum()) / 10_000

print(f"{boundary.iloc[0].get('name', REGION_NAME)}: {area_ha:,.0f} ha ({area_ha*0.00386:,.0f} sq mi)")
print(f"BBox: {tuple(round(v, 3) for v in BBOX)}")
print("\nNote: this is the Census AIANNH administrative boundary. It is not the same")
print("thing as treaty territory. The 1868 Fort Laramie Treaty territory is vastly")
print("larger. An analysis that equates the reservation polygon with 'the area of")
print("Tribal concern' has made a substantive claim it probably did not intend.")

In [ ]:
import requests
r = requests.get(
    "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/AIANNHA/MapServer/2/query",
    params={
        "where":          "UPPER(NAME) LIKE '%PINE RIDGE%'",
        "outFields":      "NAME,GEOID",
        "f":              "geojson",
        "returnGeometry": "true",
        "outSR":          "4326",
    }, timeout=30
)
print(r.status_code)
data = r.json()
print(f"Features: {len(data.get('features', []))}")
if data.get('features'):
    print("First feature geometry:", data['features'][0].get('geometry'))
else:
    print(data)

## The satellite-era fire record

MTBS perimeters, 1984 to present, clipped to the boundary.

Fire **rotation** rather than fire return interval. Rotation is the time needed
to burn an area equal to the study area, which is a landscape property computable from
mapped perimeters. Fire return interval is a point property (how often *this
spot* burns) and estimating it requires tree-ring or charcoal evidence. The two
differ by a large factor wherever fire sizes are skewed, which is everywhere,
and conflating them is a common and consequential error.

In [ ]:
fires = ta.get_fire_history(ctx, BBOX, start_year=1984, end_year=2024, min_acres=500)
fires = gpd.clip(fires, boundary) if not fires.empty else fires
print(f"{len(fires)} MTBS fires >= 500 acres within the boundary, 1984-2024")

if not fires.empty:
    fires["area_ha"] = fires.to_crs(epsg=5070).area / 10_000
    burned_by_year = fires.groupby("fire_year")["area_ha"].sum().to_dict()

    rotation = ti.fire_rotation(total_area_ha=area_ha, burned_area_by_year=burned_by_year)
    print("\nFire rotation, 1984-2024:")
    for k, v in rotation.items():
        print(f"  {k:<24} {v}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
boundary.boundary.plot(ax=axes[0], color="black", lw=1.2)
if not fires.empty:
    fires.plot(ax=axes[0], column="fire_year", cmap="inferno", alpha=0.7, legend=True)
axes[0].set_title(f"MTBS fires >= 500 ac, 1984-2024\n{REGION_NAME}")

years = range(1984, 2025)
series = pd.Series({y: burned_by_year.get(y, 0.0) for y in years}) if not fires.empty else pd.Series(0.0, index=years)
axes[1].bar(series.index, series.values, color="#b5443a")
axes[1].set_ylabel("hectares burned"); axes[1].set_title("Annual burned area")
plt.tight_layout()
plt.savefig("../outputs/01_fire_history.png", dpi=150)
plt.show()

## Ignitions and cause

Short's Fire Occurrence Database (RDS-2013-0009.6), 1992–2020. Unlike MTBS this
captures small fires, which is where most of the fire-response story lives as the
ignitions that never became large are evidence of successful initial attack, and
they are invisible in a perimeter dataset.

**Read the cause codes skeptically.** NWCG statistical cause is assigned by the
reporting agency; "Missing data/not specified/undetermined" is typically a large
share; and reporting practice varies systematically between BIA, Tribal, county,
and federal reporters and sometimes within a single reservation. A difference in
cause attribution across a jurisdictional boundary may be a reporting artifact
rather than a behavioural one, and reading it as behavioural is precisely how a
deficit narrative gets manufactured out of a data limitation.

In [ ]:
ignitions = ta.get_ignitions(ctx, BBOX, start_year=1992, end_year=2020)
ignitions = gpd.clip(ignitions, boundary) if len(ignitions) else ignitions
print(f"{len(ignitions)} FOD ignition records within the boundary, 1992-2020")

profile = ti.ignition_profile(ignitions)
print(f"\nUndetermined/missing cause: {profile.attrs.get('undetermined_pct')}% of all ignitions")
print("Any cause-based conclusion has to be stated against that fraction.\n")
profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

top = profile.head(8)
axes[0].barh(top.index.astype(str), top["n_ignitions"], color="#8c6d4f")
axes[0].set_xlabel("ignitions"); axes[0].set_title("Ignitions by reported cause, 1992-2020")
axes[0].invert_yaxis()

by_year = ignitions.groupby("fire_year").size()
axes[1].plot(by_year.index, by_year.values, "o-", ms=3, color="#3b6ea5")
axes[1].set_ylabel("ignitions/year"); axes[1].set_title("Ignitions per year")

if "fire_size" in ignitions.columns:
    axes[2].hist(np.log10(ignitions["fire_size"].clip(lower=0.1)), bins=40, color="#b5443a")
    axes[2].set_xlabel("log10(fire size, acres)")
    axes[2].set_title("Fire size distribution\n(most ignitions stay small)")

plt.tight_layout()
plt.savefig("../outputs/01_ignitions.png", dpi=150)
plt.show()

small = float((ignitions["fire_size"] < 10).mean()) if "fire_size" in ignitions.columns else np.nan
print(f"{small:.1%} of ignitions stayed under 10 acres -- that is the initial-attack record,")
print("and it is invisible in MTBS, which only maps fires above 500-1,000 acres.")

## The suppression-era baseline problem

The satellite-era rotation computed in Step 1 is set against a pre-suppression
reference fire return interval. For northern Great Plains mixed-grass prairie
and ponderosa savanna, fire-scar chronologies and LANDFIRE Biophysical Settings
generally indicate frequent fire, often on the order of every 5 to 25 years,
with grassland at the shorter end.

**What a large ratio does and does not establish.** It establishes a fire
deficit relative to the reference regime, and therefore that the satellite
record is not a natural baseline. It does *not* establish that the landscape is
unhealthy, and it does not license the conclusion that more contemporary fire
would restore the reference condition. Fires under today's fuels and climate
are not the fires that regime describes.

Replace the reference value with local fire-scar work wherever it exists. The
LANDFIRE BpS number is a modelled continental product and local chronologies
beat it every time.

In [ ]:
# Mixed-grass prairie/ponderosa savanna reference. Replace with local fire-scar
# chronology values where available. LANDFIRE BpS is modelled and continental.
REFERENCE_FRI_YEARS = 15

comparison = ti.suppression_era_comparison(
    observed_rotation_years=rotation["rotation_years"],
    reference_fri_years=REFERENCE_FRI_YEARS,
)
for k, v in comparison.items():
    print(f"{k}:\n  {v}\n" if k == "caution" else f"{k:<26} {v}")

### What the satellite record cannot see

- **Cultural burning** produced frequent, low-intensity, patchy fire across
  these landscapes for millennia. It leaves little signature that MTBS could
  detect even if MTBS had existed, because it was mostly below the mapping
  threshold, which is a statement about the instrument, not about the practice.
- **The gap is not natural.** Suppression policy and the criminalization of
  Indigenous burning interrupted a management regime. The low fire frequency in
  the early record is the effect of that interruption.
- **Knowledge of that regime persists** in communities and is not in any
  federal dataset. Where fire moved, which draws carried it, what was burned and
  when, that record exists, held by people, and it is more detailed for these
  landscapes than anything in this notebook.
- **Which means the highest-value next step is not more data.** It is a
  conversation with knowledge holders and the Tribal fire program, and an
  agreement about whether and how any of it should inform published work.

In [ ]:
acknowledgment = sv.data_acknowledgment(
    nation=NATION,
    sources=["MTBS (USGS/USFS), 1984-2024",
             "Short, K.C. 2022. Spatial wildfire occurrence data for the United States, "
             "1992-2020. FPA_FOD_20221014. Forest Service Research Data Archive. RDS-2013-0009.6",
             "U.S. Census Bureau TIGERweb AIANNH boundaries"],
)
print(acknowledgment)
print("\n" + "-" * 76 + "\n")
print(sv.ocap_note())

with open("../outputs/01_acknowledgment.txt", "w") as fh:
    fh.write(acknowledgment + "\n\n" + sv.ocap_note() + "\n")

## Summary

Fire perimeters and rotation for the satellite era, ignition records with the
undetermined-cause fraction reported, and the rotation set against a
pre-suppression reference with the limits of that comparison stated.

**The finding that matters is methodological.** The 1984–present record is not a
natural baseline; it is a record of a suppression regime layered over the
interrupted practice of cultural burning. Any subsequent analysis that treats it
as "historical range of variability" inherits that error.

Modules 2 and 3 use `fires`, `ignitions`, and `boundary`. Module 4 sets out what
would be required before any of this is published.